# 05 GWR Exploration

            This notebook prepares and interprets the geographically weighted regression model:
            `LST = beta0 + beta1(NDVI) + beta2(NDBI) + beta3(Population Density) + beta4(Built-up Density) + error`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "notebooks" / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd / "notebooks"))
else:
    raise FileNotFoundError("Could not find notebook_helpers.py. Run this notebook from the project root or notebooks folder.")

from notebook_helpers import (
    PROJECT_ROOT,
    add_project_root_to_path,
    command_string,
    find_files,
    load_yaml_config,
    notebook_metadata,
    path_status,
    plot_raster,
    print_path_status,
    project_path,
    raster_info,
    raster_stats,
    read_vector,
    run_command,
)

add_project_root_to_path()
RUN_COMMANDS = False  # Change to True only when you want notebook cells to execute CLI scripts.
YEAR = 2023
notebook_metadata("GWR Exploration", YEAR)

## Step 1 - Check model inputs

Population and built-up rasters may come from WorldPop/GHSL prep scripts or your own aligned rasters.

In [ ]:
import pandas as pd

            model_inputs = {
                "LST": f"data/processed/lst/lst_ibadan_{YEAR}_celsius.tif",
                "NDVI": f"data/processed/indices/ndvi_{YEAR}.tif",
                "NDBI": f"data/processed/indices/ndbi_{YEAR}.tif",
                "population density": f"data/processed/vulnerability/population_density_{YEAR}.tif",
                "built-up density": f"data/processed/vulnerability/built_up_density_{YEAR}.tif",
            }
            pd.DataFrame(path_status(model_inputs))

## Step 2 - Preview supporting data prep commands

In [ ]:
run_command(["python", "scripts/02b_prepare_worldpop.py", "--year", YEAR], dry_run=True)
            run_command(["python", "scripts/02c_prepare_ghsl.py", "--year", YEAR], dry_run=True)

## Step 3 - Run GWR

In [ ]:
run_command([
                "python", "scripts/07_run_gwr.py",
                "--year", YEAR,
                "--dependent", "lst",
                "--predictors", "ndvi", "ndbi", "population_density", "built_up_density",
            ], dry_run=not RUN_COMMANDS)

## Step 4 - Inspect the model grid and results

In [ ]:
gwr_path = project_path(f"data/processed/gwr/gwr_results_{YEAR}.gpkg")
            if gwr_path.exists():
                gwr = read_vector(gwr_path)
                display(gwr.head())
                print(gwr.columns.tolist())
            else:
                print("GWR results not found yet.")

## Step 5 - Read model summary

In [ ]:
summary_path = project_path(f"data/processed/gwr/gwr_summary_{YEAR}.txt")
            if summary_path.exists():
                print(summary_path.read_text(encoding="utf-8")[:4000])
            else:
                print("GWR summary not found yet.")

## Step 6 - Map local coefficients and local R2

In [ ]:
if gwr_path.exists():
                for column in ["coef_ndvi", "coef_ndbi", "coef_population_density", "coef_built_up_density", "local_r2", "gwr_residual"]:
                    if column in gwr.columns:
                        ax = gwr.plot(column=column, legend=True, figsize=(8, 7), cmap="coolwarm", edgecolor="none")
                        ax.set_title(column)
                        ax.set_axis_off()